# Visualizations for Influenza Outbreak Event Prediction via Twitter Data

Data sourced from https://archive.ics.uci.edu/dataset/861/influenza+outbreak+event+prediction+via+twitter:

Info given on UC Irvine verbatim: 

"By identifying influenza-related tweets, the goal is to forecast the spatiotemporal patterns of influenza outbreaks for different locations and dates."

"The data is from the United States. The data comes from different states under different weeks. For each week, the task is to predict whether or not there is an influenza outbreak on the next date. More specifically, for influenza activity, there are four levels of flu activities from minimal to high according to CDC Flu Activity Map. An influenza outbreak occurrence is indicated if the activity level is high."

(see https://www.semanticscholar.org/paper/SimNest%3A-Social-Media-Nested-Epidemic-Simulation-Zhao-Chen/9f5ced0409dd869890cb152cb66e06539f95daa7

for more info straight from the reasearchers)

In [45]:
from scipy.io import loadmat
import numpy as np
import pandas as pd

data = loadmat('./influenza_outbreak_dataset.mat')

# print(data.keys())
for key, val in data.items():
    if not key.startswith('__'):  # filter out metadata keys, these are generated by matlab and dont contain useful info for us
        print(f"Variable: {key} | Type: {type(val)} | Shape: {getattr(val, 'shape', 'N/A')}")

# examining shape of training and validation data
X_te_0 = data['flu_X_te'][0, 0]
Y_te_0 = data['flu_Y_te'][0, 0]

print("\nTrain Features (X_tr) shape:", X_te_0.shape)  # e.g., (N_test_days, 545)
print("Train Target (Y_tr) shape:", Y_te_0.shape)      # e.g., (N_test_days, 1)

print("\nTest Features (X_te) shape:", X_te_0.shape)  
print("Test Target (Y_te) shape:", Y_te_0.shape)     
#  

Variable: flu_X_tr | Type: <class 'numpy.ndarray'> | Shape: (1, 48)
Variable: flu_Y_tr | Type: <class 'numpy.ndarray'> | Shape: (1, 48)
Variable: flu_X_te | Type: <class 'numpy.ndarray'> | Shape: (1, 48)
Variable: flu_Y_te | Type: <class 'numpy.ndarray'> | Shape: (1, 48)
Variable: flu_locs | Type: <class 'numpy.ndarray'> | Shape: (1, 48)
Variable: flu_keywords | Type: <class 'numpy.ndarray'> | Shape: (1, 525)

Train Features (X_tr) shape: (485, 545)
Train Target (Y_tr) shape: (485, 1)

Test Features (X_te) shape: (485, 545)
Test Target (Y_te) shape: (485, 1)


As we can see our variables are tracking across 48 states, and have been stored as arrays in the matrix. 

Additionally, the data from over 3 years (1095 days) of observations have already been seperated in _tr for training and _te for testing. 

flu_locs simply stores state names. 

The keywords were used to identify tweets for a state in a given week that qualify as a record, these make up 525 of our features and the additional 20 are 

The idea is to predict based on keyword frequencies on a weekly basis how likely it is for a flu outbreak to occur. 

In [61]:
# Decoding location names and keywords
loc_names = [str(x[0]) if isinstance(x, np.ndarray) else str(x) for x in data['flu_locs'].ravel()]
keywords = [str(w[0]) if isinstance(w, np.ndarray) else str(w) for w in data['flu_keywords'].ravel()]

print(f"First 5 regions: {loc_names[:5]}")
print(f"Total regions: {len(loc_names)}, Total keywords: {len(keywords)}")

# Extract features and targets for region 0
X_region_0 = data['flu_X_tr'][0, 0]
Y_region_0 = data['flu_Y_tr'][0, 0]

# If X is sparse, convert to dense for inspection; 
# if dense, check shape
# They're all increcibly sparse b/c a tweet will only have a few of the many keywords
if hasattr(X_region_0, 'toarray'):
    X_region_0 = X_region_0.toarray()

print("Region 0 Features (X) shape:", X_region_0.shape)  # typically (T_weeks, 525)
print("Region 0 Target (Y) shape:", Y_region_0.shape)      # typically (T_weeks, 1)

First 5 regions: ['wyoming', 'colorado', 'nebraska', 'washington', 'rhode island']
Total regions: 48, Total keywords: 525
Region 0 Features (X) shape: (1095, 545)
Region 0 Target (Y) shape: (1095, 1)


# Algorithm 1 Naive Bayes

We'll take the location list and find NY to run our first algorithm on. 

In [63]:
ny_idx = next(i for i, name in enumerate(loc_names) if 'new york' in name or name == 'ny')
print(f"New York index: {ny_idx}")  

New York index: 17


With this, we can extract the neccecary training and validation data from the matrix for NY state

In [73]:
ny_idx = 17

def prep(mat):
    arr = mat[0, ny_idx]
    return arr.toarray() if hasattr(arr, 'toarray') else np.asarray(arr)

# No scaling required for GaussianNB — raw continuous values are preserved
X_train = prep(data['flu_X_tr'])[:, :525]
y_train = (prep(data['flu_Y_tr']).ravel() > 0.0).astype(int)

X_test = prep(data['flu_X_te'])[:, :525]
y_test = (prep(data['flu_Y_te']).ravel() > 0.0).astype(int)

# Check class distributions
print(f"y_train counts: Inactive={np.sum(y_train == 0)}, Active={np.sum(y_train == 1)}")
print(f"y_test counts:  Inactive={np.sum(y_test == 0)}, Active={np.sum(y_test == 1)}")


y_train counts: Inactive=1081, Active=14
y_test counts:  Inactive=429, Active=56


I suspect the class imbalance will likely affect results here. 

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.naive_bayes import GaussianNB


# No scaling required, raw continuous values can stay as is
# we only want the 525 keyword features
X_train = prep(data['flu_X_tr'])[:, :525]
y_train = (prep(data['flu_Y_tr']).ravel() > 0.0).astype(int)

X_test = prep(data['flu_X_te'])[:, :525]
y_test = (prep(data['flu_Y_te']).ravel() > 0.0).astype(int)

# Fit Gaussian Naive Bayes
gnb = GaussianNB()
gnb.fit(X_train, y_train)

# Predict and evaluate
y_pred = gnb.predict(X_test)

print("\nGaussianNB - NY Test Set")
print(classification_report(y_test, y_pred, target_names=['Inactive (0)', 'Active Flu (1)'], zero_division=0))

print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred))


GaussianNB - NY Test Set
                precision    recall  f1-score   support

  Inactive (0)       0.88      1.00      0.94       429
Active Flu (1)       0.00      0.00      0.00        56

      accuracy                           0.88       485
     macro avg       0.44      0.50      0.47       485
  weighted avg       0.78      0.88      0.83       485

Confusion Matrix
[[429   0]
 [ 56   0]]


The model is predicting 0 for all states due to how rare the success class is in the data, reducing the amount of features (choosing the top 100 from the 525) could improve this. Implementing fixed 50/50 priors and var smoothing did not improve the model's prediction ability.

# Algorithm 2